# Bước 1 — OCR sách Hán bằng PaddleOCR-VL (Baidu, 0.9B)

**Input:** các file PDF scan (*Add Input* → dataset chứa PDF). **Output:** `/kaggle/working/txt/<tên sách>.txt`
(mỗi PDF 1 file) và `txt.zip`.

**Settings:** Accelerator **GPU T4 x2**, Internet **On** → *Run All*. Hai GPU chia nhau các trang.

In [ ]:
ENGINE = "paddle"
PAGES = ""          # "" = cả sách; "1-10" = chạy thử 10 trang đầu mỗi PDF
N_LAYOUT = 3        # số trang vẽ bounding box layout
BUDGET_H = 10.5     # Kaggle cắt phiên ở 12 giờ -> dừng OCR trước để kịp ghi kết quả

# Tham số ảnh hưởng chất lượng OCR (xem README)
DPI, MAX_SIDE = 200, 1600   # độ phân giải render trang; cạnh dài tối đa của ảnh đưa vào model
MAX_NEW_TOKENS = 6000       # số token tối đa model sinh cho 1 trang

PADDLE_VERSION = "v1.6"     # hoặc "v1.5"
PADDLE_ARGS = {}            # tham số thêm cho PaddleOCRVL, vd {"use_doc_orientation_classify": True}

## 0. Cài đặt + code OCR (không cần sửa)

In [ ]:
import subprocess, sys
from pathlib import Path
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(">", cmd[:150]); print("\n".join((r.stdout + r.stderr).strip().splitlines()[-3:]))

sh(f"{sys.executable} -m pip install -q pypdfium2 uv")
VENV = "/tmp/venv_paddle"  # venv riêng: paddle dùng thư viện CUDA khác torch của Kaggle
if not Path(VENV).exists():
    sh(f"{sys.executable} -m uv venv --seed --python 3.11 {VENV}")
PY = f"{VENV}/bin/python"
sh(f'{PY} -m pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ '
   f'--extra-index-url https://pypi.org/simple')
sh(f'{PY} -m pip install -q "paddleocr[doc-parser]" pypdfium2 pillow')
sh(f'{PY} -c "import paddle; paddle.utils.run_check()"')

In [ ]:
%%writefile /kaggle/working/ocr_worker.py
"""OCR từng trang PDF (HunyuanOCR hoặc PaddleOCR-VL) trên 1 GPU; chạy lại thì bỏ qua trang đã xong."""
from __future__ import annotations

import argparse
import json
import re
import signal
import tempfile
import time
from pathlib import Path

HUNYUAN_PROMPT = ("提取文档图片中正文的所有信息用markdown格式表示，其中页眉、页脚部分忽略，"
                  "表格用html格式表达，文档中公式用latex格式表示，按照阅读顺序组织进行解析。")
PADDLE_IGNORE = {"header", "header_image", "footer", "footer_image", "number", "footnote", "aside_text",
                 "image", "chart", "seal"}


def clean_degenerate(s: str, max_unit: int = 40, min_chars: int = 60, min_cycles: int = 4) -> str:
    """Cắt đuôi lặp vô hạn (VLM kẹt vào 1 chu kỳ ngắn rồi lặp tới hết token)."""
    st = s.rstrip()
    for u in range(1, max_unit + 1):
        unit = st[-u:]
        if len(st) < min_chars or not unit.strip():
            continue
        n, i = 0, len(st)
        while i >= u and st[i - u:i] == unit:
            n, i = n + 1, i - u
        if n * u >= min_chars and n >= min_cycles:
            return st[:i].rstrip()
    return s


def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub(r"</t[dh]>", " ", md)
    md = re.sub(r"</tr>|<br\s*/?>", "\n", md)
    md = re.sub(r"<[^>]+>", "", md)
    md = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", md)
    out = []
    for ln in md.splitlines():
        ln = re.sub(r"^\s{0,3}#{1,6}\s*", "", ln)
        ln = re.sub(r"^\s*[-*+]\s+(?=\S)", "", ln)
        ln = re.sub(r"(?:[ \t,，]*\(\d{1,4},\d{1,4}\),\(\d{1,4},\d{1,4}\))+\s*$", "", ln)  # toạ độ hình của Hunyuan
        ln = ln.replace("**", "").replace("__", "").strip()
        if ln.startswith("|") and ln.endswith("|"):
            ln = " ".join(c.strip() for c in ln.strip("|").split("|") if c.strip())
        if ln and not re.fullmatch(r"[-|:\s]+", ln):
            out.append(ln)
    return out


def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if "blocks" not in rec:  # hunyuan: text markdown của cả trang
        return md_to_lines(rec["text"])
    ordered = sorted((b for b in rec["blocks"] if b["order"] is not None), key=lambda b: b["order"])
    center = lambda b: ((b["box"][0] + b["box"][2]) / 2, (b["box"][1] + b["box"][3]) / 2)  # noqa: E731
    for b in (b for b in rec["blocks"] if b["order"] is None):  # bảng, chú thích hình: paddle không xếp thứ tự đọc
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)  # đặt cạnh vùng gần nhất
    return [ln for b in ordered for ln in md_to_lines(b["text"])]


_DOCS = {}


def render(src: str, page: int, dpi: int = 200, max_side: int = 1600):
    """Trang PDF -> ảnh PIL (cạnh dài tối đa max_side)."""
    import pypdfium2 as pdfium
    if src not in _DOCS:
        _DOCS[src] = pdfium.PdfDocument(src)
    img = _DOCS[src][page].render(scale=dpi / 72).to_pil().convert("RGB")
    if max(img.size) > max_side:
        s = max_side / max(img.size)
        img = img.resize((int(img.width * s), int(img.height * s)))
    return img


class Hunyuan:
    def __init__(self, a):
        import torch
        from transformers import AutoProcessor
        try:
            from transformers import HunYuanVLForConditionalGeneration as M
        except ImportError:
            from transformers import AutoModelForImageTextToText as M
        self.torch, self.max_new, self.rep = torch, a.max_new, a.rep_penalty
        self.prompt = a.prompt or HUNYUAN_PROMPT
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[a.dtype]
        self.proc = AutoProcessor.from_pretrained(a.model, trust_remote_code=True, use_fast=False)
        self.model = M.from_pretrained(a.model, dtype=dtype, trust_remote_code=True).to(self.dev).eval()

    def run(self, img_path):
        torch = self.torch
        msgs = [{"role": "user", "content": [{"type": "image", "image": img_path},
                                             {"type": "text", "text": self.prompt}]}]
        inputs = self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt").to(self.dev)
        for k, v in list(inputs.items()):
            if torch.is_tensor(v) and v.is_floating_point():
                inputs[k] = v.to(self.model.dtype)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new, do_sample=False,
                                      repetition_penalty=self.rep)
        gen = out[:, inputs["input_ids"].shape[1]:]
        text = self.proc.batch_decode(gen, skip_special_tokens=True)[0]
        return clean_degenerate(text), {"hit_max": int(gen.shape[1]) >= self.max_new}


class Paddle:
    def __init__(self, a):
        from paddleocr import PaddleOCRVL
        self.pipe = PaddleOCRVL(pipeline_version=a.paddle_version, **json.loads(a.paddle_args))
        self.max_new = a.max_new

    def run(self, img_path):
        try:
            res = list(self.pipe.predict(img_path, max_new_tokens=self.max_new))[0]
        except TypeError:  # bản paddleocr cũ chưa nhận max_new_tokens
            res = list(self.pipe.predict(img_path))[0]
        return self.parse(res.json)

    @staticmethod
    def parse(j):
        """res.json của PaddleOCR-VL -> (text, {"blocks": vùng đã đọc chữ, "boxes": mọi khung layout})."""
        j = j.get("res", j) if isinstance(j, dict) else {}
        blocks = [{"label": b.get("block_label"), "box": [round(x) for x in b.get("block_bbox", [0, 0, 0, 0])],
                   "order": b.get("block_order"), "text": clean_degenerate(b.get("block_content") or "")}
                  for b in j.get("parsing_res_list", []) if b.get("block_label") not in PADDLE_IGNORE]
        blocks = [b for b in blocks if b["text"]]
        text = "\n".join(page_lines({"blocks": blocks}))
        boxes = [{"label": b["label"], "box": [round(x) for x in b["coordinate"]], "order": b.get("order"),
                  "score": round(b.get("score", 0), 2)}
                 for b in (j.get("layout_det_res") or {}).get("boxes", [])]
        return text, {"blocks": blocks, "boxes": boxes}


ENGINES = {"hunyuan": Hunyuan, "paddle": Paddle}


def result_file(out: Path, book: str, name: str, shard: int) -> Path:
    return out / book / f"{name}_shard{shard}.jsonl"


def load_results(out: Path, book: str, name: str) -> dict[int, dict]:
    """{trang: bản ghi} của 1 sách, gộp mọi shard (dòng hỏng thì bỏ)."""
    recs = {}
    for f in sorted((out / book).glob(f"{name}_shard*.jsonl")):
        for line in open(f, encoding="utf-8"):
            try:
                r = json.loads(line)
                recs[r["page"]] = r
            except (json.JSONDecodeError, KeyError):
                pass
    return recs


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--engine", required=True, choices=list(ENGINES))
    ap.add_argument("--name", default=None, help="tên file kết quả (mặc định = engine)")
    ap.add_argument("--tasks", required=True, help='jsonl, mỗi dòng {"book", "src", "page"}')
    ap.add_argument("--out", required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshard", type=int, default=1)
    ap.add_argument("--dpi", type=int, default=200)
    ap.add_argument("--max-side", type=int, default=1600)
    ap.add_argument("--max-new", type=int, default=6000)
    ap.add_argument("--dtype", default="fp32", choices=["fp32", "fp16", "bf16"])
    ap.add_argument("--model", default="tencent/HunyuanOCR")
    ap.add_argument("--prompt", default="", help="hunyuan: prompt OCR thay cho prompt mặc định")
    ap.add_argument("--rep-penalty", type=float, default=1.08)
    ap.add_argument("--paddle-version", default="v1.6")
    ap.add_argument("--paddle-args", default="{}", help="json tham số thêm cho PaddleOCRVL")
    ap.add_argument("--limit", type=int, default=None, help="chỉ chạy N trang (chạy thử)")
    ap.add_argument("--page-timeout", type=int, default=300, help="giây tối đa/trang, quá thì bỏ trang đó")
    a = ap.parse_args()
    name, out = a.name or a.engine, Path(a.out)

    tasks = [json.loads(l) for l in open(a.tasks, encoding="utf-8") if l.strip()]
    mine = [t for i, t in enumerate(tasks) if i % a.nshard == a.shard]
    done, skip = set(), set()
    for book in {t["book"] for t in mine}:
        done |= {(book, p) for p in load_results(out, book, name)}
        for f in (out / book).glob(f"{name}_skip*.log"):
            skip |= {(book, int(p)) for p in f.read_text().split()}
    # trang đang chạy dở khi worker bị giết vì treo -> ghi vào skip, không thử lại
    inflight = out / f"_inflight_{name}_{a.shard}"
    if inflight.exists():
        book, _, p = inflight.read_text().rpartition("|")
        if book and (book, int(p)) not in done:
            with open(out / book / f"{name}_skip.log", "a") as f:
                f.write(p + "\n")
            skip.add((book, int(p)))
            print(f"[{name}#{a.shard}] {book} trang {int(p) + 1}: bị treo ở lần trước -> bỏ qua", flush=True)
        inflight.unlink()
    seen = done | skip
    todo = [t for t in mine if (t["book"], t["page"]) not in seen][:a.limit]
    print(f"[{name}#{a.shard}] {len(mine)} trang, đã xong {len(mine) - len(todo)}, chạy {len(todo)}", flush=True)
    if not todo:
        return

    engine = ENGINES[a.engine](a)
    tmp = Path(tempfile.mkdtemp()) / "page.png"

    def on_timeout(signum, frame):
        raise TimeoutError(f"quá {a.page_timeout}s")
    signal.signal(signal.SIGALRM, on_timeout)

    t_start = time.time()
    for n, t in enumerate(todo, 1):
        book, page, t0 = t["book"], t["page"], time.time()
        (out / book).mkdir(parents=True, exist_ok=True)
        inflight.write_text(f"{book}|{page}")
        signal.alarm(a.page_timeout)
        try:
            img = render(t["src"], page, a.dpi, a.max_side)
            img.save(tmp)
            text, extra = engine.run(str(tmp))
            signal.alarm(0)
            rec = {"page": page, "text": text, "w": img.width, "h": img.height,
                   "sec": round(time.time() - t0, 1), **extra}
            with open(result_file(out, book, name, a.shard), "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            eta = (time.time() - t_start) / n * (len(todo) - n) / 60
            print(f"[{name}#{a.shard}] {n}/{len(todo)} {book} tr.{page + 1}: {rec['sec']}s, "
                  f"{len(text)} ký tự | còn ~{eta:.0f} phút", flush=True)
        except Exception as e:  # noqa: BLE001 - 1 trang lỗi không làm dừng cả lượt
            signal.alarm(0)
            if isinstance(e, TimeoutError):
                with open(out / book / f"{name}_skip.log", "a") as f:
                    f.write(f"{page}\n")
            print(f"[{name}#{a.shard}] {book} tr.{page + 1}: LỖI {type(e).__name__}: {str(e)[:200]}", flush=True)
    inflight.unlink(missing_ok=True)
    for d in _DOCS.values():  # đóng PDF, khỏi in cảnh báo pypdfium2 vào log
        d.close()


if __name__ == "__main__":
    main()

## 1. Tìm PDF → danh sách trang

In [ ]:
import glob, json, os, re, shutil, time
from pathlib import Path
import pypdfium2 as pdfium
sys.path.insert(0, "/kaggle/working")
import ocr_worker as ow

N_GPU = len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines())
if N_GPU == 0:
    raise SystemExit("!!! Không có GPU: Settings -> Accelerator -> GPU T4 x2 rồi Run All lại.")
WORK = Path("/kaggle/working"); RAW = WORK / "ocr_raw"; TXT = WORK / "txt"
RAW.mkdir(exist_ok=True); TXT.mkdir(exist_ok=True)

def parse_pages(spec, n):
    """'1-10,15' -> [0..9, 14] (đếm từ 1); '' -> mọi trang."""
    if not spec.strip():
        return list(range(n))
    out = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        out += range(int(a) - 1, min(int(b or a), n))
    return sorted(set(out))

books = {}  # tên sách -> (đường dẫn PDF, các trang cần OCR)
for p in sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True)):
    if "__MACOSX" in p:
        continue
    b = re.sub(r"[^\w\-]+", "_", Path(p).stem)
    while b in books:
        b += "_"
    books[b] = (p, parse_pages(PAGES, len(pdfium.PdfDocument(p))))
    print(f"{b:<50} {len(books[b][1]):5d} trang")
assert books, "Không thấy PDF nào trong /kaggle/input -- đã Add Input chưa?"

TASKS = str(WORK / "tasks.jsonl")
with open(TASKS, "w", encoding="utf-8") as f:
    for b, (src, sel) in books.items():
        for pg in sel:
            f.write(json.dumps({"book": b, "src": src, "page": pg}, ensure_ascii=False) + "\n")
print(f"Tổng {sum(len(s) for _, s in books.values())} trang, {N_GPU} GPU")

# Chạy tiếp lần trước: Add Input = Output của phiên trước -> trang đã OCR không chạy lại
n_prev = 0
for f in glob.glob("/kaggle/input/**/ocr_raw/*/*", recursive=True):
    f = Path(f)
    if f.parent.name in books and f.name.startswith(ENGINE) and f.suffix in (".jsonl", ".log"):
        dst = RAW / f.parent.name / (f.stem + "_prev" + f.suffix)
        if not dst.exists():
            dst.parent.mkdir(exist_ok=True); shutil.copy(f, dst); n_prev += 1
print(f"Nạp lại {n_prev} file kết quả cũ")

ENGINE_ARGS = ["--paddle-version", PADDLE_VERSION, "--paddle-args", json.dumps(PADDLE_ARGS)]

def run_ocr(tasks, name=ENGINE, extra=(), limit=None, budget_h=BUDGET_H, stall_min=15):
    """Mỗi GPU 1 worker, chia đều trang; log đứng yên quá stall_min phút -> khởi động lại worker."""
    jobs = []
    for k in range(N_GPU):
        cmd = [PY, "/kaggle/working/ocr_worker.py", "--engine", ENGINE, "--name", name, "--tasks", tasks,
               "--out", str(RAW), "--shard", str(k), "--nshard", str(N_GPU), "--dpi", str(DPI),
               "--max-side", str(MAX_SIDE), "--max-new", str(MAX_NEW_TOKENS), *ENGINE_ARGS, *extra]
        cmd += ["--limit", str(limit)] if limit else []
        log = RAW / f"_{name}_{k}.log"
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(k)}
        jobs.append({"cmd": cmd, "env": env, "log": log, "restart": 0,
                     "p": subprocess.Popen(cmd, stdout=open(log, "a"), stderr=subprocess.STDOUT, env=env)})
    t0 = t_print = time.time()
    while any(j["p"].poll() is None for j in jobs):
        time.sleep(30)
        if time.time() - t0 > budget_h * 3600:
            print(f"!!! Hết {budget_h} giờ -> dừng OCR, phần đã xong vẫn được ghi ra txt")
            for j in jobs:
                j["p"].terminate()
            break
        for j in jobs:
            if j["p"].poll() is None and time.time() - os.path.getmtime(j["log"]) > stall_min * 60 and j["restart"] < 5:
                j["p"].kill(); j["p"].wait(); j["restart"] += 1
                j["p"] = subprocess.Popen(j["cmd"], stdout=open(j["log"], "a"), stderr=subprocess.STDOUT, env=j["env"])
                print(f"!!! {j['log'].name}: treo -> khởi động lại, bỏ trang đang kẹt")
        if time.time() - t_print > 120:
            t_print = time.time()
            last = [(open(j["log"], errors="ignore").read().strip().splitlines() or ["..."])[-1][-120:] for j in jobs]
            print(f"[{(time.time() - t0) / 60:5.1f} phút] " + " | ".join(last), flush=True)
    for j in jobs:
        if j["p"].wait() not in (0, -15):
            print(f"\n!!! {j['log'].name} lỗi, 20 dòng cuối log:")
            print("\n".join(open(j["log"], errors="ignore").read().splitlines()[-20:]))

## 2. Layout detection
PaddleOCR-VL chạy 2 bước: model layout (PP-DocLayout) khoanh các vùng + thứ tự đọc, rồi model VL đọc chữ từng vùng — vẽ các khung của bước 1.

In [ ]:
from PIL import ImageDraw, ImageFont
from IPython.display import display
COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#00a0b0", "#f032e6", "#9a6324", "#808000", "#000075"]

def demo_pages(n):
    """n trang ở giữa các sách (trang bìa ít khung, xem không thú vị)."""
    out = []
    for frac in (1 / 3, 1 / 2, 2 / 3, 1 / 4, 3 / 4):
        for b, (src, sel) in books.items():
            t = {"book": b, "src": src, "page": sel[int(len(sel) * frac)]} if sel else None
            if t and t not in out and len(out) < n:
                out.append(t)
    return out

def draw_boxes(t, boxes):
    """Vẽ [(nhãn, [x1, y1, x2, y2])] (toạ độ pixel ảnh đã render) lên trang, đánh số theo thứ tự đọc."""
    img = ow.render(t["src"], t["page"], DPI, MAX_SIDE)
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, img.width // 60))
    labels = sorted({l for l, _ in boxes})
    for i, (label, box) in enumerate(boxes, 1):
        c = COLORS[labels.index(label) % len(COLORS)]
        d.rectangle(box, outline=c, width=3)
        d.text((box[0] + 4, box[1] + 2), f"{i}.{label}", fill=c, font=font)
    (WORK / "layout").mkdir(exist_ok=True)
    img.save(WORK / "layout" / f"{ENGINE}_{t['book']}_tr{t['page'] + 1}.png")
    print(f"{t['book']} — trang {t['page'] + 1}: {len(boxes)} khung: " + ", ".join(f"{i}.{l}" for i, (l, _) in enumerate(boxes, 1)))
    display(img.resize((700, int(700 * img.height / img.width))))

demo = demo_pages(N_LAYOUT)
with open(WORK / "demo.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in demo)

run_ocr(str(WORK / "demo.jsonl"), budget_h=1)  # các trang này OCR luôn, lượt chạy chính sẽ bỏ qua
for t in demo:
    r = ow.load_results(RAW, t["book"], ENGINE).get(t["page"])
    if r:
        boxes = sorted(r["boxes"], key=lambda b: (b["order"] is None, b["order"] or 0))
        draw_boxes(t, [(b["label"], b["box"]) for b in boxes])

## 3. Chạy thử 1 trang mỗi GPU

In [ ]:
run_ocr(TASKS, limit=1, budget_h=1)
for b in books:
    for r in list(ow.load_results(RAW, b, ENGINE).values())[:1]:
        print(f"--- {b} — trang {r['page'] + 1} ({r['sec']}s) ---\n{r['text'][:500]}\n")

## 4. OCR toàn bộ

In [ ]:
run_ocr(TASKS)

## 5. Ghi ra txt

In [ ]:
for b, (src, sel) in books.items():
    recs = ow.load_results(RAW, b, ENGINE)
    pages = ["\n".join(ow.page_lines(recs[p])) for p in sel if p in recs]
    (TXT / f"{b}.txt").write_text("\n\n".join(p for p in pages if p) + "\n", encoding="utf-8")
    miss = len(sel) - len(pages)
    print(f"{b:<50} {len(pages)}/{len(sel)} trang" + (f"  -- THIẾU {miss} trang" if miss else ""))
shutil.make_archive(str(WORK / "txt"), "zip", TXT)
print("\nKết quả: /kaggle/working/txt/*.txt và txt.zip. Thiếu trang (hết giờ / lỗi): *Add Input* = Output của phiên "
      "này rồi Run All lại, trang đã xong không chạy lại.")